In [3]:
!pip install "transformers==4.44.2" "huggingface_hub==0.23.5" --upgrade


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [31 lines of output]
  Python reports SOABI: cp313-win_amd64
  Computed rustc target triple: x86_64-pc-windows-msvc
  Installation directory: C:\Users\moham\AppData\Local\puccinialin\puccinialin\Cache
  Checking for Rust toolchain....
  Rust not found, installing into a temporary directory
  
  Installing rust to C:\Users\moham\AppData\Local\puccinialin\puccinialin\Cache\rustup
  warn: installing msvc toolchain without its prerequisites
  info: profile set to 'minimal'
  info: default host triple is x86_64-pc-windows-msvc
  info: syncing channel updates for 'stable-x86_64-pc-windows-msvc'
  info: latest update on 2025-11-10, rust version 1.91.1 (ed61e7d7e 2025-11-07)
  info: downloading component 'cargo'
  info: downloading component 'rust-std'
  info: downloading component 'rustc'
  info: installing component 'cargo'
  info: installing component 'rust-std'
  inf

In [4]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import torch


In [5]:
dataset = load_dataset("ag_news")

train_ds = dataset["train"]   # 120,000 samples
test_ds  = dataset["test"]    # 7,600 samples

num_labels = 4  # AG News has 4 categories
print(train_ds[0])


{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


### BERT - Model Training

### Tokenization and Data Preprocessing
We are doing the following transformations on the data:
Now each example has:
- input_ids: [101, 2023, 2003, ...] / after converting each word of the text into id's
- attention_mask: [1, 1, 1, 0, 0, ...] / essentially making only the words of the text to have the attention and paddings added to the text to become umimportant for the classification.
- label: 0–3

In [7]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

max_len = 200

def preprocess(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",  # fixed length = max_len
        max_length=max_len
    )

train_ds_enc = train_ds.map(preprocess, batched=True)
test_ds_enc  = test_ds.map(preprocess, batched=True)

Map: 100%|████████████████████████████████████████████████████████████████| 7600/7600 [00:02<00:00, 3202.90 examples/s]


In [8]:
train_ds_enc = train_ds_enc.remove_columns(["text"])
test_ds_enc  = test_ds_enc.remove_columns(["text"])

In [9]:
train_ds_enc.set_format("torch")
test_ds_enc.set_format("torch")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


In [10]:
torch.cuda.is_available()

False